# AICIDENTIA - 05 RQ3: risk-domain classification and label audit

**RQ3:** How accurately can incident text be classified into risk domains, and how far do the AI-generated (MIT) risk labels agree with human judgement?

> CRISP-DM phases: **Modeling** (classifiers) and **Evaluation** (nested cross-validation, bootstrap CIs, paired tests, calibration, kappa); **Deployment** feeds the Streamlit app

Logic lives in `aicidentia.models.classify`, `classify_outputs`, `label_audit` and `aicidentia.evaluation.metrics`; the write-ups are `docs/rq3_classification.md` and, once you have annotated, `docs/rq3_label_audit.md`. **The target (MIT Risk Domain) is machine-generated**: classifier accuracy shows how well text reproduces that labeller, not agreement with humans.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from aicidentia import registry
from aicidentia.config import PROJECT_ROOT, configure_logging, load_config, set_seeds

configure_logging()
cfg = load_config()
SEED = set_seeds(cfg["random_seed"])
FIG = PROJECT_ROOT / cfg["paths"]["figures"]
pd.set_option("display.width", 220, "display.max_colwidth", 90)
print(f"{cfg['project_name']} | seed {SEED} | snapshot {cfg['snapshot_date']}")

## 1. Evaluate every classifier

Folds are fixed once and saved for the Colab run; hyperparameters are tuned by an inner 3-fold CV on training data only (nested CV). Everything is cached in `data/interim/`, so this takes seconds after the first run.

In [ ]:
from aicidentia.models import classify_outputs

res = classify_outputs.run_classification()
classify_outputs.write_all(res)
f = res.facts
print(f["labelled"], "labelled incidents;", len(f["classes"]), "classes;", f["class_counts"])
print(
    "time split train/test:",
    f["time_train"],
    "/",
    f["time_test"],
    "| DistilBERT scored:",
    res.colab_present,
)

## 2. Results: macro-F1 with bootstrap confidence intervals

Macro-F1 is primary because the classes are very unequal (a majority-class model gets about 38% accuracy but a macro-F1 near 0.08).

In [ ]:
display(
    res.metrics[
        ["model", "setting", "n", "macro_f1", "ci_low", "ci_high", "weighted_f1", "accuracy"]
    ].round(3)
)
display(Image(filename=FIG / "fig_rq3_1_model_comparison.png"))

**Paired comparisons** (same incidents, paired bootstrap of the macro-F1 difference and exact McNemar test). Most differences between the real models are within noise.

In [ ]:
display(
    res.significance[
        ["model_a", "model_b", "setting", "diff", "ci_low", "ci_high", "p_value", "mcnemar_p"]
    ].round(4)
)
print("best cross-validated model:", res.best_model)
print("deployable:", res.deployable, "|", res.deployable_reason)

## 3. Which domains are hard?

In [ ]:
d = res.data
support = pd.Series({c: int((d.y == i).sum()) for i, c in enumerate(d.classes)})
pc = res.per_class[
    res.per_class["class"].isin(d.classes)
    & ~res.per_class["model"].str.contains("ablation|majority")
]
display(
    pc.pivot_table(index="class", columns="model", values="f1")
    .round(3)
    .reindex(support.sort_values(ascending=False).index)
)
display(Image(filename=FIG / "fig_rq3_3_per_class_f1.png"))
display(Image(filename=FIG / "fig_rq3_2_confusion.png"))
display(res.confusions_top.round(3))

## 4. Calibration of the confidence shown to users

Three options compared on out-of-fold predictions (calibrators fitted inside each training part): uncalibrated, Platt (sigmoid) and temperature scaling. Platt made things *worse* here; temperature scaling is selected by the pre-specified rule.

In [ ]:
cal = res.calibration
print(
    pd.DataFrame(
        {
            m: {
                k: round(cal[m][k], 4)
                for k in ["ece", "brier", "accuracy", "mean_confidence", "macro_f1"]
            }
            for m in ["raw", "sigmoid", "temperature"]
        }
    )
)
print("selected:", cal["chosen"])
display(Image(filename=FIG / "fig_rq3_4_reliability.png"))

## 5. Highest-weight terms per domain and the CSET proxies

In [ ]:
display(res.features[res.features["model"] == classify_outputs.BASELINE][["domain", "top_terms"]])
display(res.cset["intent"])
display(res.cset["protected"])

## 6. DistilBERT on Colab

`notebooks/colab_distilbert_finetune.ipynb` uses identical folds. Copy its output to `reports/colab/rq3_distilbert_predictions.csv` and re-run this notebook (or `scripts/run_phase6_classification.py`); the model is then scored in the tables above whether or not it beats the baseline.

In [ ]:
colab = PROJECT_ROOT / "reports" / "colab"
print({p.name: p.stat().st_size for p in colab.glob("*.csv")})
print("DistilBERT predictions present:", res.colab_present)

## 7. Label audit: your blind annotation

150 incidents, stratified by MIT domain with a floor of 10 per domain, shuffled, with the MIT label **not** in the file. Fill `my_label` and `my_confidence` in `data/manual/label_audit_sample.csv` (guidelines: `docs/label_guidelines.md`; helper: `uv run streamlit run scripts/label_audit_app.py`), then run `uv run python scripts/run_phase6_audit.py`.

In [ ]:
from aicidentia.models import label_audit as A

display(res.audit["allocation"])
print("annotation progress:", A.annotation_progress(classes=res.data.classes))

In [ ]:
from aicidentia.models import label_audit_outputs as audit_out

result = audit_out.run_audit_analysis()
if result is None:
    print(
        "No annotations yet: nothing to analyse. (Kappa, confusion matrix and per-domain agreement appear here once you annotate.)"
    )
else:
    audit_out.write_all(result, res.data.classes)
    print(
        {
            k: result[k]
            for k in [
                "n_annotated",
                "complete",
                "agreement_raw",
                "kappa_raw",
                "kappa_raw_ci",
                "kappa_raw_band",
                "kappa_weighted",
                "kappa_weighted_ci",
                "kappa_weighted_band",
            ]
        }
    )
    display(result["confusion"])
    display(result["per_domain"].round(3))

## 8. Registry

In [ ]:
for key in [
    "rq3.best_model",
    "rq3.deployable",
    "rq3.calibration_method",
    "rq3.audit_sample_status",
]:
    print(key, "=", registry.get(key))